# Hierarchical Token Tree Visualization

This notebook visualizes the sampled hierarchical token collections in `hierarchical_chinese_web_token_tree_sample.jsonl`.

It uses only Python standard-library modules plus `IPython.display`, so reviewers can run it in a regular Jupyter environment without installing Graphviz or other visualization packages.


In [ ]:
import json
import html
from pathlib import Path
from IPython.display import HTML, display

DATA_PATH = Path("hierarchical_chinese_web_token_tree_sample.jsonl")
if not DATA_PATH.exists():
    DATA_PATH = Path("anonymous_review_release_20260525/hierarchical_chinese_web_token_tree_sample.jsonl")

records = [
    json.loads(line)
    for line in DATA_PATH.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

by_collection_id = {record["collection_id"]: record for record in records}
by_tree_id = {record["tree_id"]: record for record in records}
by_root_token = {record["root"]["token"].strip(): record for record in records}

len(records), DATA_PATH


## Collection Index

Run the next cell to see the 20 sampled tree collections and their basic metadata.


In [ ]:
import json
import html
from pathlib import Path
from IPython.display import HTML, display

if "records" not in globals():
    DATA_PATH = Path("hierarchical_chinese_web_token_tree_sample.jsonl")
    if not DATA_PATH.exists():
        DATA_PATH = Path("anonymous_review_release_20260525/hierarchical_chinese_web_token_tree_sample.jsonl")
    records = [
        json.loads(line)
        for line in DATA_PATH.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    by_collection_id = {record["collection_id"]: record for record in records}
    by_tree_id = {record["tree_id"]: record for record in records}
    by_root_token = {record["root"]["token"].strip(): record for record in records}


def esc(value):
    return html.escape(str(value), quote=True)


def visible_token(token):
    token = str(token)
    leading_spaces = len(token) - len(token.lstrip(" "))
    if leading_spaces:
        return "[space]" * leading_spaces + token[leading_spaces:]
    return token


def count_nodes(node):
    return 1 + sum(count_nodes(child) for child in node.get("children", []))


def tree_max_depth(node):
    children = node.get("children", [])
    if not children:
        return node.get("depth", 0)
    return max(tree_max_depth(child) for child in children)


def collection_index_html():
    rows = []
    for record in records:
        tree = record["hierarchical_tree"]
        rows.append(
            "<tr>"
            f"<td>{esc(record['collection_id'])}</td>"
            f"<td>{esc(record['tree_id'])}</td>"
            f"<td><code>{esc(visible_token(record['root']['token']))}</code></td>"
            f"<td><span class='label-pill'>{esc(record['target_label'])}</span></td>"
            f"<td>{esc(record['token_counts']['full_tree'])}</td>"
            f"<td>{esc(tree_max_depth(tree))}</td>"
            "</tr>"
        )
    return """
    <style>
      .tree-table-card {
        display: inline-block;
        overflow: hidden;
        border: 1px solid #dbeafe;
        border-radius: 10px;
        box-shadow: 0 10px 24px rgba(15, 23, 42, 0.10);
        background: #ffffff;
        margin: 6px 0 18px 0;
      }
      .tree-table {
        border-collapse: separate;
        border-spacing: 0;
        font-family: ui-sans-serif, system-ui, -apple-system, Segoe UI, sans-serif;
        font-size: 14px;
        color: #172033;
        min-width: 760px;
      }
      .tree-table th {
        background: linear-gradient(90deg, #0f766e, #2563eb);
        color: #ffffff;
        padding: 9px 10px;
        text-align: left;
        font-weight: 700;
        border: 0;
      }
      .tree-table td {
        padding: 8px 10px;
        border-top: 1px solid #e5e7eb;
        background: #ffffff;
      }
      .tree-table tr:nth-child(even) td { background: #f8fafc; }
      .tree-table tr:hover td { background: #ecfeff; }
      .tree-table code {
        background: #e0f2fe;
        color: #0f172a;
        padding: 2px 6px;
        border-radius: 6px;
        font-weight: 600;
      }
      .label-pill {
        display: inline-block;
        background: #eef2ff;
        color: #3730a3;
        padding: 2px 7px;
        border-radius: 999px;
        font-size: 12px;
        font-weight: 650;
      }
    </style>
    <div class="tree-table-card">
      <table class="tree-table">
        <thead>
          <tr>
            <th>collection_id</th><th>tree_id</th><th>root token</th><th>target label</th>
            <th>tokens</th><th>max depth</th>
          </tr>
        </thead>
        <tbody>
    """ + "\n".join(rows) + """
        </tbody>
      </table>
    </div>
    """


def show_collection_index():
    display(HTML(collection_index_html()))


show_collection_index()


## Tree Viewer

Use `display_tree(...)` with a `tree_id`, `collection_id`, or root token. Set `max_depth=None` to expand the full tree.


In [ ]:
TREE_CSS = """
<style>
  .tree-wrap {
    font-family: ui-sans-serif, system-ui, -apple-system, Segoe UI, sans-serif;
    line-height: 1.5;
    color: #172033;
    background: #f8fafc;
    border: 1px solid #dbeafe;
    border-radius: 12px;
    padding: 18px 20px;
    box-shadow: 0 12px 28px rgba(15, 23, 42, 0.10);
  }
  .tree-title {
    margin: 0 0 8px 0;
    font-size: 21px;
    font-weight: 750;
    color: #0f172a;
  }
  .tree-meta {
    margin: 0 0 16px 0;
    color: #475569;
    font-size: 13px;
  }
  .composition {
    margin: 12px 0 18px 0;
    padding: 12px 14px;
    background: linear-gradient(135deg, #ecfeff, #eef2ff);
    border: 1px solid #67e8f9;
    border-left: 5px solid #0f766e;
    border-radius: 10px;
    color: #164e63;
    box-shadow: inset 0 1px 0 rgba(255, 255, 255, 0.75);
  }
  .node, .leaf { margin: 5px 0 5px 16px; }
  .children {
    margin-left: 17px;
    border-left: 2px solid #bfdbfe;
    padding-left: 10px;
  }
  summary { cursor: pointer; }
  .token {
    font-family: ui-monospace, SFMono-Regular, Consolas, monospace;
    background: #ffffff;
    color: #0f172a;
    border: 1px solid #cbd5e1;
    padding: 2px 6px;
    border-radius: 7px;
    font-weight: 650;
    box-shadow: 0 1px 2px rgba(15, 23, 42, 0.05);
  }
  .label {
    display: inline-block;
    color: #075985;
    background: #e0f2fe;
    margin-left: 6px;
    padding: 1px 7px;
    border-radius: 999px;
    font-size: 12px;
    font-weight: 700;
  }
  .meta {
    color: #64748b;
    margin-left: 6px;
    font-size: 12px;
  }
  .not-covered > summary, .not-covered { opacity: .72; }
  .exclude {
    color: #991b1b;
    background: #fee2e2;
    margin-left: 6px;
    padding: 1px 6px;
    border-radius: 999px;
    font-size: 12px;
  }
</style>
"""


def resolve_record(key):
    if key in by_collection_id:
        return by_collection_id[key]
    if key in by_tree_id:
        return by_tree_id[key]
    if isinstance(key, str) and key.strip() in by_root_token:
        return by_root_token[key.strip()]
    raise KeyError(f"No collection found for {key!r}")


def node_html(node, max_show_depth=4):
    children = node.get("children", [])
    depth = node.get("depth", 0)
    token = esc(visible_token(node.get("token", "")))
    label = esc(node.get("label", ""))
    reason_covered = node.get("reason_covered", True)
    coverage_class = "" if reason_covered else " not-covered"
    excluded = ""
    if node.get("exclude_reason"):
        excluded = f'<span class="exclude">{esc(node["exclude_reason"])}</span>'
    summary = (
        f'<span class="token">{token}</span>'
        f'<span class="label">{label}</span>'
        f'<span class="meta">depth={esc(depth)}, children={esc(len(children))}</span>'
        f'{excluded}'
    )
    can_expand = children and (max_show_depth is None or depth < max_show_depth)
    if can_expand:
        rendered_children = "".join(node_html(child, max_show_depth) for child in children)
        return f'<details open class="node{coverage_class}"><summary>{summary}</summary><div class="children">{rendered_children}</div></details>'
    if children:
        summary += '<span class="meta">collapsed below max_depth</span>'
    return f'<div class="leaf{coverage_class}">{summary}</div>'


def composition_html(record):
    composition = record.get("composition")
    if not composition:
        return ""
    parts = []
    for item in composition.get("composition_tokens", []):
        parts.append(
            f'<span class="token">{esc(visible_token(item["token"]))}</span>'
            f'<span class="label">{esc(item["label"])}</span>'
        )
    left = " + ".join(parts)
    right = f'<span class="token">{esc(visible_token(composition["composite_token"]))}</span>'
    interpretation = esc(composition.get("interpretation", ""))
    return f'<div class="composition"><b>Composition:</b> {left} &rarr; {right}<br>{interpretation}</div>'


def display_tree(key, max_depth=4):
    record = resolve_record(key)
    tree = record["hierarchical_tree"]
    header = f"""
    <div class="tree-wrap">
      <div class="tree-title"><code>{esc(visible_token(record['root']['token']))}</code> ({esc(record['target_label'])})</div>
      <div class="tree-meta">
        collection_id={esc(record['collection_id'])}; tree_id={esc(record['tree_id'])};
        tokens={esc(record['token_counts']['full_tree'])}; max_depth={esc(tree_max_depth(tree))};
        representative_token=<code>{esc(visible_token(record['representative_token']))}</code>
      </div>
      {composition_html(record)}
      {node_html(tree, max_depth)}
    </div>
    """
    display(HTML(TREE_CSS + header))


## Examples

The following examples show two composition cases. Change the argument or `max_depth` to inspect other collections.


In [ ]:
display_tree(11366, max_depth=4)


In [ ]:
display_tree(23984, max_depth=3)
